In [ ]:
# pip install nltk newspaper3k scikit-learn numpy

# Import libraries
from newspaper import Article
import random
import string
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Download required NLTK tokenizers (both punkt and punkt_tab for compatibility)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# 1. Fetch and process the article corpus
url = 'https://www.mayoclinic.org/diseases-conditions/chronic-kidney-disease/symptoms-causes/syc-20354521'
try:
    article = Article(url)
    article.download()
    article.parse()
    article.nlp()
    corpus = article.text
except Exception as e:
    print(f"Warning: Could not fetch from URL ({e}). Using sample fallback corpus.")
    corpus = (
        "Chronic kidney disease, also called chronic kidney failure, involves a gradual loss of kidney function. "
        "Your kidneys filter wastes and excess fluids from your blood, which are then removed in your urine. "
        "Advanced chronic kidney disease can cause dangerous levels of fluid, electrolytes and wastes to build up in your body. "
        "Signs and symptoms of chronic kidney disease develop over time if kidney damage progresses slowly. "
        "Loss of kidney function can cause a buildup of fluid or body waste products or electrolyte problems. "
        "Symptoms include nausea, vomiting, loss of appetite, fatigue and weakness, sleep problems, and changes in how much you urinate."
    )

# Tokenize corpus into a list of sentences
text = corpus
sentence_list = nltk.sent_tokenize(text)


# Helper function: Sort indices based on similarity scores in descending order
# (This function was missing from the original slides, causing NameError: index_sort)
def index_sort(list_var):
    length = len(list_var)
    list_index = list(range(0, length))
    for i in range(length):
        for j in range(length):
            if list_var[list_index[i]] > list_var[list_index[j]]:
                temp = list_index[i]
                list_index[i] = list_index[j]
                list_index[j] = temp
    return list_index


# Bot response generation based on cosine similarity
def bot_response(user_input):
    user_input = user_input.lower()
    sentence_list.append(user_input)
    bot_reply = ''

    cm = CountVectorizer().fit_transform(sentence_list)
    similarity_scores = cosine_similarity(cm[-1], cm)
    similarity_scores_lists = similarity_scores.flatten()

    # Sort similarity scores descending
    index = index_sort(similarity_scores_lists)
    index = index[1:]  # Exclude the user's own input (highest similarity = 1.0)
    response_flag = 0

    j = 0
    for i in range(len(index)):
        if similarity_scores_lists[index[i]] > 0.0:
            bot_reply = bot_reply + ' ' + sentence_list[index[i]]
            response_flag = 1
            j = j + 1
        if j > 2:
            break

    if response_flag == 0:
        bot_reply = bot_reply + ' ' + "I apologize, I don't understand."

    # Remove the appended user input safely
    sentence_list.pop()

    return bot_reply.strip()


# Function to return a random response to user greetings
def greeting_response(text):
    text = text.lower()
    bot_greetings = ["hii", "hey", "hello", "hola"]
    user_greeting = ['hi', 'hey', 'hello', 'hola', 'greetings']

    for word in text.split():
        clean_word = word.strip(string.punctuation)
        if clean_word in user_greeting:
            return random.choice(bot_greetings)
    return None


# Chatbot interaction loop
def start_chat():
    print("Doc Bot : I am doc bot for your services for a short time. I will answer your questions.")
    print("(Type 'exit', 'bye', or 'quit' to end the conversation.)\n")

    exit_list = ['exit', 'see you later', 'bye', 'quit', 'break']

    while True:
        try:
            user_input = input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nDoc Bot : Chat with you later!")
            break

        if not user_input:
            continue

        if user_input.lower() in exit_list:
            print("Doc Bot : Chat with you later!")
            break
        else:
            greeting = greeting_response(user_input)
            if greeting is not None:
                print('Doc Bot: ' + greeting)
            else:
                print('Doc Bot: ' + bot_response(user_input))


if __name__ == "__main__":
    start_chat()
